# Test the Account Statement API

Calls the one-call statement API (`POST /api/statement`, deployed on Vercel) and saves the returned PDF and CSV.

1. Run the cells in order. The credentials cell asks for the **API URL** (`https://<project>.vercel.app/api/statement`) and an **API token** with hidden prompts, so neither is saved in the notebook.
2. Call `get_statement(user_id, date_from, date_to)` with a user ID or email. Dates are `YYYY-MM-DD`; leave them out for the full history (first transaction to today).
3. The PDF and CSV are decoded from base64 into a `statements/` folder, with download links shown under the cell.

Ask the project owner for a token. Clear outputs before committing: they contain the full statement.

In [ ]:
import base64, json
from getpass import getpass
from pathlib import Path

import requests
from IPython.display import FileLink, display

In [ ]:
def get_statement(user_id, date_from=None, date_to=None, out_dir="statements"):
    """Request a statement and save the PDF and CSV. Dates are 'YYYY-MM-DD' or None (full history)."""
    body = {"userId": user_id}
    if date_from: body["from"] = date_from
    if date_to:   body["to"] = date_to

    resp = requests.post(API_URL, json=body,
                         headers={"Authorization": f"Bearer {TOKEN}"}, timeout=90)
    data = resp.json()

    if resp.status_code != 200:
        print(f"HTTP {resp.status_code}: {data.get('message') or data.get('error')}")
        if "issues" in data:
            print(json.dumps(data["issues"], indent=2))
        return None

    out = Path(out_dir); out.mkdir(exist_ok=True)
    pdf_path = out / data["fileName"]
    csv_path = out / data["csv"]["fileName"]
    pdf_path.write_bytes(base64.b64decode(data["contentBase64"]))
    csv_path.write_bytes(base64.b64decode(data["csv"]["contentBase64"]))

    period, meta = data["period"], data["meta"]
    print(f"Period: {period['from']} → {period['to']}"
          + ("  (full history)" if period["fullHistory"] else ""))
    print(f"Transactions: {meta['output_txn_count']} of {meta['input_txn_count']}"
          + ("  (truncated, request a shorter period)" if meta["is_truncated"] else ""))
    display(FileLink(pdf_path), FileLink(csv_path))   # clickable download links
    return data

In [ ]:
#Credentials
API_URL = getpass("API URL: ")
TOKEN = getpass("API token: ")          # hidden input, not saved in the notebook

In [ ]:
# Test Run
get_statement("barrie.morris@example.com")